# Task 1: Exploratory Data Analysis — Brent Oil Prices

**Project:** Birhan Energies — Change Point Analysis  
**Objective:** Investigate time series properties (trend, stationarity, volatility) to inform Bayesian change point modeling.

## Contents
1. Data loading and overview
2. Price series visualization
3. Log returns and volatility analysis
4. Stationarity testing (ADF)
5. Event overlay analysis
6. Key findings and modeling implications

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_brent_prices, load_events, compute_log_returns
from src.analysis import adf_test, summary_by_period, event_window_stats

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

## 1. Data Loading and Overview

In [ ]:
prices = load_brent_prices()
events = load_events()
returns = compute_log_returns(prices)

print(f"Observations: {len(prices):,}")
print(f"Date range:   {prices.index.min().date()} → {prices.index.max().date()}")
print(f"Price range:  ${prices['price'].min():.2f} – ${prices['price'].max():.2f}")
print(f"Mean price:   ${prices['price'].mean():.2f}")
print(f"\nEvents loaded: {len(events)}")
prices.head()

## 2. Price Series with Key Events

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(prices.index, prices["price"], linewidth=0.8, color="#2c3e50")

colors = {"Conflict": "#e74c3c", "OPEC": "#3498db", "Economic": "#f39c12", "Geopolitical": "#9b59b6"}
for _, ev in events.iterrows():
    ax.axvline(ev["date"], color=colors.get(ev["category"], "#95a5a6"), alpha=0.4, linewidth=0.8)

ax.set_title("Brent Crude Oil Prices (1987–2022) with Key Market Events", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD/barrel)")
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

## 3. Log Returns and Volatility Clustering

Log returns $r_t = \log(P_t) - \log(P_{t-1})$ are used because raw prices are non-stationary. Volatility clustering (periods of high/low variance) is visible during crises: 2008, 2014–2016, 2020, 2022.

In [ ]:
print(f"Mean daily return:  {returns.mean()*100:.4f}%")
print(f"Std daily return:   {returns.std()*100:.4f}%")
print(f"Annualized vol:     {returns.std() * np.sqrt(252) * 100:.2f}%")
print(f"Skewness:           {returns.skew():.3f}")
print(f"Kurtosis:           {returns.kurtosis():.3f}  (excess; Normal = 0)")

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(returns.index, returns.values, linewidth=0.5, color="#2980b9", alpha=0.8)
axes[0].set_title("Daily Log Returns")
axes[0].set_ylabel("Log Return")
axes[0].axhline(0, color="black", linewidth=0.5)

rolling_vol = returns.rolling(30).std() * np.sqrt(252) * 100
axes[1].plot(rolling_vol.index, rolling_vol.values, linewidth=1, color="#e74c3c")
axes[1].set_title("30-Day Rolling Annualized Volatility (%)")
axes[1].set_ylabel("Volatility (%)")
axes[1].set_xlabel("Date")
plt.tight_layout()
plt.show()

## 4. Stationarity Testing

The Augmented Dickey-Fuller (ADF) test checks whether a unit root is present (non-stationary). We test both raw prices and log returns.

In [ ]:
adf_prices = adf_test(prices["price"], "Prices")
adf_returns = adf_test(returns, "Log Returns")

for r in [adf_prices, adf_returns]:
    print(f"\n{r['name']}:")
    print(f"  ADF statistic: {r['adf_statistic']:.4f}")
    print(f"  p-value:       {r['p_value']:.6f}")
    print(f"  Result:        {r['interpretation']}")

## 5. Decade Summary and Event Window Analysis

In [ ]:
print("=== Summary by Decade ===")
display(summary_by_period(prices))

print("\n=== Event Window Stats (±30 days) ===")
event_stats = event_window_stats(prices, events)
display(event_stats.sort_values("pct_change", key=abs, ascending=False))

## 6. Key Findings and Modeling Implications

### Time Series Properties

| Property | Finding | Modeling Implication |
|----------|---------|---------------------|
| **Trend** | Strong non-stationary trend; prices rose from ~$18 (1987) to peaks above $140 (2008, 2022) | Apply change point model to **log returns**, not raw prices |
| **Stationarity** | ADF p=0.26 for prices (non-stationary); p≈0.0 for log returns (stationary) | Log returns are the appropriate target variable |
| **Volatility** | Clear clustering during 2008, 2014–2016, 2020, 2022 crises; annualized vol ≈ 41% | Consider variance change point models as extension |
| **Distribution** | Fat tails (excess kurtosis > 0); non-Normal | Normal likelihood is simplifying assumption; Student-t as extension |

### Visually Identified Regime Shifts

1. **1990–1991:** Gulf War spike ($17 → $40+)
2. **1998:** Asian crisis collapse (below $10)
3. **2003–2008:** Sustained bull market ($25 → $147 peak)
4. **2008–2009:** Financial crisis crash ($147 → $40)
5. **2011:** Arab Spring elevated plateau ($100+)
6. **2014–2016:** OPEC price war ($110 → $45)
7. **2020:** COVID demand collapse ($50 → $20)
8. **2022:** Ukraine war spike ($80 → $128)

### Next Steps (Task 2)

- Build single change point PyMC model on log returns
- Compare posterior τ with event dates from `oil_market_events.csv`
- Quantify before/after mean shifts with probabilistic statements
- Extend to multiple change points for major regime shifts identified above